In [5]:
import jax
import jax.numpy as jnp

# 模拟 self.state.target_params —— 定义在函数外面
target_w = jnp.array(2.0)

def critic_loss(online_w, rng):       # 只有 online_w 是形参
    target_q = target_w * 10          # 闭包里的 target_w → 常数
    pred_q = online_w * 10            # 形参 online_w → 求导变量
    return (pred_q - target_q) ** 2

online_w = jnp.array(3.0)
print(jax.grad(critic_loss)(online_w, jax.random.PRNGKey(0)))
# 输出: 200.0
# 解释: loss = (30-20)^2=100, d/ d(online_w) = 2*(30-20)*10 = 200
# target_w=2 参与算 loss，但 grad 里没有 target_w —— 因为对 online_w 求导时它是常数
def critic_loss2(online_w, target_w_arg, rng):
    target_q = target_w_arg * 10
    pred_q = online_w * 10
    return (pred_q - target_q) ** 2

# 现在可以对 target 求导了（但 SAC 故意不这么做）：
print(jax.grad(critic_loss2, argnums=1)(online_w, target_w, jax.random.PRNGKey(0)))
# 输出: -200

200.0
-200.0


In [16]:
import jax
import jax.numpy as jnp

# 一个有「双层结构」的 pytree,模拟 (grads, aux) 二元组
template = {"a": lambda x: x, "b": lambda x: x}   # 当模板:dict 形状
data = {
    "a": (jnp.array([1.]), jnp.array([2.])),
    "b": (jnp.array([3.]), jnp.array([4.])),
}

# 正确:用 template 划定叶子位置
# print(jax.tree.map(lambda _, x: x[1], template, data))
# 输出: {a: [2.], b: [4.]}

# 错误:只传 data,jax 会一路展开到数组
print(jax.tree.map(lambda x: x[1], data))
# 报错:TypeError: only integer scalar arrays can be converted to a scalar index
import jax
print(jax.__version__)



{'a': (Array(1., dtype=float32), Array(2., dtype=float32)), 'b': (Array(3., dtype=float32), Array(4., dtype=float32))}
0.4.35


In [11]:
# jax.tree_map(fn, *trees) 有一个很多人初次接触都会踩坑的硬规矩:它以「第一个 pytree 的结构」为模板,确定「叶子在哪里」,然后只对后续 pytree 扁平化到对应深度。​
# 把它想成「第一个 pytree 划定了一层『皮』,后续 pytree 就在这层皮下面被原样塞进去」。所以 lambda 收到的不是「最底层的标量」,而是「皮下面那一整块对象」。
# treemap第一个是模板
import jax
import jax.numpy as jnp

# loss_fns: 两个叶子 callable
loss_fns = {
    "actor":  lambda p, r: jnp.sum(p["w1"] * r[0]),   # 占位
    "critic": lambda p, r: jnp.sum(p["w2"] * r[1]),
}

# 假设 grads_and_aux 在 has_aux=True 时是这种结构:
# 外层 dict 包了两个叶子,每个叶子是 (grad_pytree, aux_pytree) 二元组
grads_and_aux = {
    "actor": (
        {"w1": jnp.array([1.0])},          # grad tree
        {"entropy": jnp.array([0.5])}      # aux tree
    ),
    "critic": (
        {"w2": jnp.array([2.0])},
        {"td_error": jnp.array([0.3])}
    ),
}

# 源码写法
aux = jax.tree.map(lambda _, x: x[1], loss_fns, grads_and_aux)
print(aux)
# 你的提议: 只传一个 pytree,简化 lambda
aux = jax.tree.map(lambda x: x[1], grads_and_aux)
print(aux)
# 为什么对?​
# jax.tree_map 看到第一个参数 loss_fns 是 dict,
# 叶子是两个 lambda。所以它就在 dict 这一层停下,
# 不深入 lambda 内部。然后去 grads_and_aux 同位置——也就是 "actor" 那个槽位——取一整块塞给 lambda,
# 这块就是 ({"w1": ...}, {"entropy": ...}) 整个二元组。Lambda 用 x[1] 取第二个元素,即 aux tree。



{'actor': {'entropy': Array([0.5], dtype=float32)}, 'critic': {'td_error': Array([0.3], dtype=float32)}}
{'actor': ({'w1': Array(1., dtype=float32)}, {'entropy': Array(0.5, dtype=float32)}), 'critic': ({'w2': Array(2., dtype=float32)}, {'td_error': Array(0.3, dtype=float32)})}


In [7]:
import functools
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
from flax import struct
from typing import Any, Callable, Dict, Sequence, Mapping


# ---------- 复用第一部分的工具 ----------
def shard_batch(batch, sharding):
    return jax.tree_map(
        lambda x: jax.device_put(x, sharding.reshape(
            sharding.shape[0], *((1,) * (x.ndim - 1)))),
        batch,
    )


# ---------- ModuleDict(完整保留) ----------
class ModuleDict(nn.Module):
    modules: Dict[str, nn.Module]

    @nn.compact
    def __call__(self, *args, name=None, **kwargs):
        if name is None:
            if kwargs.keys() != self.modules.keys():
                raise ValueError(
                    f"kwargs={kwargs.keys()} vs modules={self.modules.keys()}")
            out = {}
            for key, value in kwargs.items():
                if isinstance(value, Mapping):
                    out[key] = self.modules[key](**value)
                elif isinstance(value, Sequence):
                    out[key] = self.modules[key](*value)
                else:
                    out[key] = self.modules[key](value)
            return out
        return self.modules[name](*args, **kwargs)


# ---------- JaxRLTrainState(完整保留) ----------
class JaxRLTrainState(struct.PyTreeNode):
    step: int
    apply_fn: Callable = struct.field(pytree_node=False)
    params: Any
    target_params: Any
    txs: Any = struct.field(pytree_node=False)
    opt_states: Any
    rng: Any
    epsilon: float = 0.0

    @staticmethod
    def _tx_tree_map(*args, **kwargs):
        return jax.tree_map(
            *args,
            is_leaf=lambda x: isinstance(x, optax.GradientTransformation),
            **kwargs,
        )

    def target_update(self, tau):
        new_target = jax.tree_map(
            lambda p, tp: p * tau + tp * (1 - tau),
            self.params, self.target_params)
        return self.replace(target_params=new_target)

    def apply_gradients(self, *, grads):
        updates_and_new_states = self._tx_tree_map(
            lambda tx, opt_state, grad:
                tx.update(grad, opt_state, self.params),
            self.txs, self.opt_states, grads)
        updates = self._tx_tree_map(
            lambda _, x: x[0], self.txs, updates_and_new_states)
        new_opt_states = self._tx_tree_map(
            lambda _, x: x[1], self.txs, updates_and_new_states)
        updates_flat = []
        self._tx_tree_map(
            lambda _, u: updates_flat.append(u), self.txs, updates)
        updates_acc = jax.tree_map(
            lambda *xs: jnp.sum(jnp.array(xs), axis=0), *updates_flat)
        new_params = optax.apply_updates(self.params, updates_acc)
        return self.replace(
            step=self.step + 1, params=new_params, opt_states=new_opt_states)

    def apply_loss_fns(self, loss_fns, pmap_axis=None, has_aux=False):
        treedef = jax.tree_util.tree_structure(loss_fns)
        new_rng, *rngs = jax.random.split(self.rng, treedef.num_leaves + 1)
        rngs = jax.tree_util.tree_unflatten(treedef, rngs)
        grads_and_aux = jax.tree_map(
            lambda fn, rng:
                jax.grad(fn, has_aux=has_aux)(self.params, rng),
            loss_fns, rngs)
        self = self.replace(rng=new_rng)
        if pmap_axis is not None:
            grads_and_aux = jax.lax.pmean(grads_and_aux, axis_name=pmap_axis)
        if has_aux:
            grads = jax.tree_map(lambda _, x: x[0], loss_fns, grads_and_aux)
            aux  = jax.tree_map(lambda _, x: x[1], loss_fns, grads_and_aux)
            return self.apply_gradients(grads=grads), aux
        return self.apply_gradients(grads=grads_and_aux)

    @classmethod
    def create(cls, *, apply_fn, params, txs, target_params=None,
               rng=jax.random.PRNGKey(0), epsilon=0.0):
        return cls(
            step=0, apply_fn=apply_fn, params=params,
            target_params=target_params, txs=txs,
            opt_states=cls._tx_tree_map(lambda tx: tx.init(params), txs),
            rng=rng, epsilon=epsilon,
        )


# ---------- 网络定义: shared encoder + actor + critic ----------
class Encoder(nn.Module):
    feat_dim: int = 32
    @nn.compact
    def __call__(self, x):
        x = nn.Dense(64)(x); x = nn.relu(x)
        x = nn.Dense(self.feat_dim)(x)
        return x

class Actor(nn.Module):
    encoder: nn.Module
    action_dim: int
    @nn.compact
    def __call__(self, obs):
        z = self.encoder(obs)
        mu  = nn.Dense(self.action_dim)(z)
        log_std = nn.Dense(self.action_dim)(z)
        return mu, log_std   # SAC 重参数化采样所需

class Critic(nn.Module):
    encoder: nn.Module     # 与 actor 共享同一个 encoder 实例
    @nn.compact
    def __call__(self, obs, action):
        z = self.encoder(obs)
        x = jnp.concatenate([z, action], axis=-1)
        q = nn.Dense(64)(x); q = nn.relu(q)
        q = nn.Dense(1)(q)
        return q.squeeze(-1)


# ---------- 训练主循环 ----------
def main():
    obs_dim, action_dim = 10, 4
    rng = jax.random.PRNGKey(0)

    encoder  = Encoder()
    actor    = Actor(encoder=encoder, action_dim=action_dim)
    critic   = Critic(encoder=encoder)
    model_def = ModuleDict({"actor": actor, "critic": critic})

    rng, init_rng = jax.random.split(rng)
    example_obs    = jnp.zeros((1, obs_dim))
    example_action = jnp.zeros((1, action_dim))
    variables = model_def.init(init_rng, actor=example_obs,
                               critic=(example_obs, example_action))
    params = variables["params"]

    # 三个独立优化器 + 三个独立 loss 函数(pytree 结构必须一致)
    txs = {
        "actor":   optax.adam(3e-4),
        "critic":  optax.adam(3e-4),
        "entropy": optax.adam(3e-4),
    }

    state = JaxRLTrainState.create(
        apply_fn=model_def.apply,
        params=params,
        txs=txs,
        target_params=params,        # 初始化 target 等于 online
        rng=rng,
    )

    # ---- 关键: 把三个 loss 写成 pytree,结构与 txs 一致 ----
    def make_loss_fns(alpha=0.2, gamma=0.99):
        def actor_loss(params, rng):
            mu, log_std = state.apply_fn(
                {"params": params}, example_obs, name="actor")
            std = jnp.exp(log_std)
            eps = jax.random.normal(rng, mu.shape)
            a   = mu + eps * std
            q   = state.apply_fn(
                {"params": params}, example_obs, action=a, name="critic")
            # SAC 温度项: log_prob = -0.5 * ((a-mu)/std)^2 - log_std
            log_prob = -0.5 * ((a - mu) / std) ** 2 - log_std
            return jnp.mean(alpha * log_prob - q)

        def critic_loss(params, rng):
            # 用 target_params 计算 TD target(冻结一份)
            q_pred = state.apply_fn(
                {"params": params}, example_obs, action=example_action,
                name="critic")
            q_tgt  = state.apply_fn(
                {"params": state.target_params}, example_obs,
                action=example_action, name="critic")
            target = example_action[:, 0] * 0.0 + 0.5   # 伪 reward
            td     = target + gamma * q_tgt - q_pred
            return jnp.mean(td ** 2)

        def entropy_loss(params, rng):
            # 训练 log_alpha 的简化版: 把 alpha 当作可学习标量参数(简化演示)
            mu, _ = state.apply_fn(
                {"params": params}, example_obs, name="actor")
            return jnp.mean(mu ** 2) * 0.0  # 占位

        return {"actor": actor_loss, "critic": critic_loss,
                "entropy": entropy_loss}

    @jax.jit
    def train_step(state, batch):
        # 把 loss pytree 在 jit 边界内闭包,但 loss 内部用的 example_obs
        # 是常数(在 jit 边界外),会被 trace 成常量。
        loss_fns = make_loss_fns()
        new_state, aux = state.apply_loss_fns(loss_fns, has_aux=True)
        new_state = new_state.target_update(tau=0.005)
        return new_state, aux

    # 伪训练循环
    for step in range(5):
        fake_batch = jnp.ones((1, obs_dim))
        state, aux = train_step(state, fake_batch)
        print(f"step={state.step}, actor_loss={float(aux['actor']):.4f},"
              f" critic_loss={float(aux['critic']):.4f}")


In [5]:
import jax
import jax.numpy as jnp
import optax
from jax.tree_util import tree_map

# 让所有非 dict/list/array 都被当叶子
def stop_at(x):
    return not isinstance(x, (dict, list, tuple)) or isinstance(x, optax.GradientTransformation)

# 1) params
params = {
    "actor":  {"act": {"kernel": jnp.zeros((32, 1)), "bias": jnp.zeros((1,))}},
    "critic": {"q":   {"kernel": jnp.zeros((33, 1)), "bias": jnp.zeros((1,))}},
}

# 2) txs(全用 adam,避开 sgd 的 EmptyState)
txs = {
    "actor":  {"act": {"kernel": optax.adam(1e-3), "bias": optax.adam(1e-3)}},
    "critic": {"q":   {"kernel": optax.adam(1e-3), "bias": optax.adam(1e-3)}},
}

# 3) opt_states
def init_one(tx, p):
    return tx.init(p)

opt_states = tree_map(init_one, txs, params, is_leaf=stop_at)

# 4) grads
grads = {
    "actor":  {"act": {"kernel": jnp.ones((32, 1))*0.1, "bias": jnp.ones((1,))*0.1}},
    "critic": {"q":   {"kernel": jnp.ones((33, 1))*0.2, "bias": jnp.ones((1,))*0.2}},
}

# 5) 一行 update:四棵树同形 + is_leaf 钩子保持 namedtuple 完整
def update_leaf(tx, opt_state, grad, p):
    updates, new_state = tx.update(grad, opt_state, p)
    return updates   # 只返回 updates,new_state 单独处理

# 关键:这里 f 返回的是 array(叶子),tree_map 能正常处理
updates_tree = tree_map(
    update_leaf, txs, opt_states, grads, params,
    is_leaf=stop_at,
)

# 6) new_params
new_params = tree_map(lambda p, u: p + u, params, updates_tree)

print("new_params['actor']['act']['kernel'] 前 2 值:")
print(new_params['actor']['act']['kernel'][:2])


new_params['actor']['act']['kernel'] 前 2 值:
[[-0.00100002]
 [-0.00100002]]


In [5]:
import jax
devices = jax.local_devices()
num_devices = len(devices)
sharding = jax.sharding.PositionalSharding(devices)


def print_green(x):
    return print("\033[92m {}\033[00m".format(x))
print_green(sharding)
print_green(num_devices)
print_green(jax.local_device_count())


 PositionalSharding([{GPU 0} {GPU 1}], memory_kind=device, shape=(2,))
 2
 2


In [1]:
import logging
from rich.logging import RichHandler

# 配置现代化的日志系统
logging.basicConfig(
    level="INFO",
    format="%(message)s",
    datefmt="[%X]",
    handlers=[RichHandler(rich_tracebacks=True)]
)

log = logging.getLogger("serl_trainer")

# 在代码中这样使用
log.info("Starting learner loop")  # 普通信息
log.warning("Grasp penalty detected!") # 警告信息（默认带黄色/橙色高亮）
log.info("[bold green]Loaded previous buffer data.[/bold green]", extra={"markup": True}) # 强制绿色


[15:39:17] INFO     Starting learner loop                                                          ]8;id=4544712;file:///tmp/ipykernel_2958281/4169173552.py\4169173552.py]8;;\:]8;id=4544713;file:///tmp/ipykernel_2958281/4169173552.py#15\15]8;;\

           WARNING  Grasp penalty detected!                                                        ]8;id=4544719;file:///tmp/ipykernel_2958281/4169173552.py\4169173552.py]8;;\:]8;id=4544720;file:///tmp/ipykernel_2958281/4169173552.py#16\16]8;;\

           INFO     Loaded previous buffer data.                                                   ]8;id=4544726;file:///tmp/ipykernel_2958281/4169173552.py\4169173552.py]8;;\:]8;id=4544727;file:///tmp/ipykernel_2958281/4169173552.py#17\17]8;;\

In [ ]:
from serl_launcher.common.common import JaxRLTrainState,ModuleDict,nonpytree_field,shard_batch
from serl_launcher.common.encoding import EncodingWrapper
import jax
import jax.numpy as jnp
from jax.sharding import PositionalSharding

# 假设有 8 块设备
sharding = PositionalSharding(jax.devices())   # shape = (8,)

batch = {
    "observations": jnp.ones((256, 17)),    # 2 维 → reshape 成 (8, 1)
    "images":       jnp.ones((256, 84, 84, 3)),  # 4 维 → reshape 成 (8,1,1,1)
    "actions":      jnp.ones((256, 6)),
}

sharded = shard_batch(batch, sharding)
# 结果:每个字段的 256 条样本被均分成 8×32,
# 每块设备各拿 32 条,后续 pmap/jit 计算自动在本地完成。
# print(jax.local_devices() == jax.devices()) # true [CudaDevice(id=0), CudaDevice(id=1)]



True
[CudaDevice(id=0), CudaDevice(id=1)]


In [7]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from collections.abc import Mapping, Sequence

class Encoder(nn.Module):
    @nn.compact
    def __call__(self, x):
        x = nn.Dense(64, name="fc1")(x)
        x = nn.relu(x)
        return nn.Dense(32, name="fc2")(x)

class Actor(nn.Module):
    encoder: nn.Module
    @nn.compact
    def __call__(self, obs):                       # 1 个位置参数
        return nn.Dense(1, name="act")(self.encoder(obs))

class Critic(nn.Module):
    encoder: nn.Module
    @nn.compact
    def __call__(self, obs, action):               # 2 个关键字参数
        x = jnp.concatenate([self.encoder(obs), action], axis=-1)
        return nn.Dense(1, name="q")(x)

class ModuleDict(nn.Module):
    modules: dict
    @nn.compact
    def __call__(self, *args, name=None, **kwargs):
        if name is None:                           # ============== init 模式 ==============
            if kwargs.keys() != self.modules.keys():
                raise ValueError(
                    f"kwargs={list(kwargs)} 与 modules={list(self.modules)} 不一致"
                )
            out = {}
            for key, value in kwargs.items():
                if isinstance(value, Mapping):      # dict → **
                    out[key] = self.modules[key](**value)
                elif isinstance(value, Sequence):   # tuple/list → *
                    out[key] = self.modules[key](*value)
                else:                              # 单个数组 → 直接当位置参数
                    out[key] = self.modules[key](value)
            return out
        return self.modules[name](*args, **kwargs)  # ===== 调用模式,按 name 转发 =====

# ---------------- 组装 ----------------
shared_encoder = Encoder()
model_def = ModuleDict({
    "actor":  Actor(encoder=shared_encoder),
    "critic": Critic(encoder=shared_encoder),
})

rng = jax.random.PRNGKey(0)
obs = jnp.zeros((1, 8))
act = jnp.zeros((1, 2))

# actor 走"else"分支(数组当位置参数),
# critic 走"Mapping"分支(字典用 ** 拆)
params = model_def.init(
    rng,
    actor=obs,
    critic={"obs": obs, "action": act},
)

# 验证 encoder 的字段只出现一次
flat = jax.tree_util.tree_flatten_with_path(params)[0]
encoder_paths = [p for p, _ in flat if "fc1" in p or "fc2" in p]
print("共享 encoder 的字段路径:", encoder_paths)
# 期望:只有 ['encoder_1/fc1', 'encoder_1/fc2'] 这一组,actor/critic 都不重复

a_out = model_def.apply(params, obs, name="actor")
q_out = model_def.apply(params, obs, action=act, name="critic")
print("actor out:", a_out.shape, "critic out:", q_out.shape)  # (1,1) (1,1)


共享 encoder 的字段路径: []
actor out: (1, 1) critic out: (1, 1)


In [8]:
import jax

def param_table(params):
    rows = jax.tree_util.tree_flatten_with_path(params)[0]
    out = []
    for path, leaf in rows:
        # path 是 (key1, key2, ...) 的元组,每个 key 可能是
        # DictKey('actor') 或 SeqKey(0) 或 'param_name' 等;
        # str(key) 对它们都能拿到形如 "['actor']" / ".0" 的片段,拼起来就成完整路径
        flat = "/".join(str(k) for k in path)
        out.append((flat, leaf.shape))
    return out

# 1) 完整路径表
for p, s in param_table(params):
    print(f"{p:<40s} {tuple(s)}")


['params']/['modules_actor']/['act']/['bias'] (1,)
['params']/['modules_actor']/['act']/['kernel'] (32, 1)
['params']/['modules_actor']/['encoder']/['fc1']/['bias'] (64,)
['params']/['modules_actor']/['encoder']/['fc1']/['kernel'] (8, 64)
['params']/['modules_actor']/['encoder']/['fc2']/['bias'] (32,)
['params']/['modules_actor']/['encoder']/['fc2']/['kernel'] (64, 32)
['params']/['modules_critic']/['q']/['bias'] (1,)
['params']/['modules_critic']/['q']/['kernel'] (34, 1)


In [13]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from collections.abc import Mapping, Sequence

# 在 Flax 内部"登记参数"这件事上加 hook,看看 init 时
# 哪些 nn.Dense 被实例化了、形状是什么
orig_dense_init = nn.Dense.__init__

def hooked_dense_init(self, *args, **kwargs):
    print(f"  [Flax 内部登记] nn.Dense(features={args[0] if args else kwargs.get('features')}, "
          f"name={kwargs.get('name', '<auto>')})")
    orig_dense_init(self, *args, **kwargs)

nn.Dense.__init__ = hooked_dense_init

# ============== 子模块定义(同前) ==============
class Encoder(nn.Module):
    @nn.compact
    def __call__(self, x):
        x = nn.Dense(64, name="fc1")(x); x = nn.relu(x)
        x = nn.Dense(32, name="fc2")(x); return x

class Actor(nn.Module):
    encoder: nn.Module
    @nn.compact
    def __call__(self, obs):
        feat = self.encoder(obs)
        return nn.Dense(1, name="act")(feat)

class Critic(nn.Module):
    encoder: nn.Module
    @nn.compact
    def __call__(self, obs, action):
        feat = self.encoder(obs)
        x = jnp.concatenate([feat, action], axis=-1)
        return nn.Dense(1, name="q")(x)

class ModuleDict(nn.Module):
    modules: dict
    @nn.compact
    def __call__(self, *args, name=None, **kwargs):
        print(f"[ModuleDict] name={name!r} kwargs={list(kwargs)}")
        if name is None:
            assert kwargs.keys() == self.modules.keys()
            out = {}
            for k, v in kwargs.items():
                if isinstance(v, Mapping):    out[k] = self.modules[k](**v)
                elif isinstance(v, Sequence): out[k] = self.modules[k](*v)
                else:                         out[k] = self.modules[k](v)
            return out
        return self.modules[name](*args, **kwargs)

# ============== 组装 + init ==============
shared_encoder = Encoder()
model_def = ModuleDict({"actor": Actor(encoder=shared_encoder),
                        "critic": Critic(encoder=shared_encoder)})
rng = jax.random.PRNGKey(0)
obs, act = jnp.zeros((1, 8)), jnp.zeros((1, 2))

print("=" * 60)
print("调用 model_def.init(...) 之前")
print("=" * 60)
params = model_def.init(rng, actor=obs, critic={"obs": obs, "action": act})
print("=" * 60)
print("init 调用完毕,params 已返回")
print("=" * 60)

# 把 params 树打散打印
print("params 树的所有叶子路径 + 形状:")
for path, leaf in jax.tree_util.tree_flatten_with_path(params)[0]:
    print(f"  {'/'.join(str(k) for k in path):<55s} {tuple(leaf.shape)}")


调用 model_def.init(...) 之前
[ModuleDict] name=None kwargs=['actor', 'critic']
  [Flax 内部登记] nn.Dense(features=64, name=fc1)
  [Flax 内部登记] nn.Dense(features=32, name=fc2)
  [Flax 内部登记] nn.Dense(features=1, name=act)
  [Flax 内部登记] nn.Dense(features=64, name=fc1)
  [Flax 内部登记] nn.Dense(features=32, name=fc2)
  [Flax 内部登记] nn.Dense(features=1, name=q)
init 调用完毕,params 已返回
params 树的所有叶子路径 + 形状:
  ['params']/['modules_actor']/['act']/['bias']           (1,)
  ['params']/['modules_actor']/['act']/['kernel']         (32, 1)
  ['params']/['modules_actor']/['encoder']/['fc1']/['bias'] (64,)
  ['params']/['modules_actor']/['encoder']/['fc1']/['kernel'] (8, 64)
  ['params']/['modules_actor']/['encoder']/['fc2']/['bias'] (32,)
  ['params']/['modules_actor']/['encoder']/['fc2']/['kernel'] (64, 32)
  ['params']/['modules_critic']/['q']/['bias']            (1,)
  ['params']/['modules_critic']/['q']/['kernel']          (34, 1)


In [9]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from collections.abc import Mapping, Sequence

# ============== 1) 子模块定义 ==============
class Encoder(nn.Module):
    @nn.compact
    def __call__(self, x):
        print(f"  [Encoder.__call__] 收到 x.shape={x.shape}")
        x = nn.Dense(64, name="fc1")(x)
        x = nn.relu(x)
        x = nn.Dense(32, name="fc2")(x)
        return x

class Actor(nn.Module):
    encoder: nn.Module
    @nn.compact
    def __call__(self, obs):
        print(f"  [Actor.__call__]   收到 obs.shape={obs.shape}")
        feat = self.encoder(obs)             # ← 关键:显式调用共享对象
        return nn.Dense(1, name="act")(feat)

class Critic(nn.Module):
    encoder: nn.Module
    @nn.compact
    def __call__(self, obs, action):
        print(f"  [Critic.__call__]  收到 obs={obs.shape}, action={action.shape}")
        feat = self.encoder(obs)             # ← 同一个 encoder 实例
        x = jnp.concatenate([feat, action], axis=-1)
        return nn.Dense(1, name="q")(x)

# ============== 2) ModuleDict 容器 ==============
class ModuleDict(nn.Module):
    modules: dict
    @nn.compact
    def __call__(self, *args, name=None, **kwargs):
        print(f"\n>>> ModuleDict.__call__  name={name!r}  "
              f"args={tuple(a.shape for a in args)}  kwargs_keys={list(kwargs)}")
        if name is None:                      # ============ init 模式 ============
            assert kwargs.keys() == self.modules.keys(), \
                "init 必须为每个子模块传一个样例输入"
            out = {}
            for key, value in kwargs.items():
                if isinstance(value, Mapping):
                    out[key] = self.modules[key](**value)
                elif isinstance(value, Sequence):
                    out[key] = self.modules[key](*value)
                else:
                    out[key] = self.modules[key](value)
            return out
        return self.modules[name](*args, **kwargs)

# ============== 3) 组装 ==============
shared_encoder = Encoder()
model_def = ModuleDict({
    "actor":  Actor(encoder=shared_encoder),
    "critic": Critic(encoder=shared_encoder),
})

rng = jax.random.PRNGKey(0)
obs = jnp.zeros((1, 8))
act = jnp.zeros((1, 2))

# ============== 4) init 阶段 ==============
print("\n========== init(name=None) ==========")
params = model_def.init(rng, actor=obs, critic={"obs": obs, "action": act})

print("\n========== params 树 ==========")
def show(paths):
    for p, leaf in jax.tree_util.tree_flatten_with_path(paths)[0]:
        print(f"  {'/'.join(str(k) for k in p):<48s} {tuple(leaf.shape)}")
show(params)

# ============== 5) apply 阶段,两个不同的 name ==============
print("\n========== apply(name='actor') ==========")
a_out = model_def.apply(params, obs, name="actor")
print("a_out.shape =", a_out.shape)

print("\n========== apply(name='critic', action=...) ==========")
q_out = model_def.apply(params, obs, action=act, name="critic")
print("q_out.shape =", q_out.shape)



========== init(name=None) ==========

>>> ModuleDict.__call__  name=None  args=()  kwargs_keys=['actor', 'critic']
  [Actor.__call__]   收到 obs.shape=(1, 8)
  [Encoder.__call__] 收到 x.shape=(1, 8)
  [Critic.__call__]  收到 obs=(1, 8), action=(1, 2)
  [Encoder.__call__] 收到 x.shape=(1, 8)

========== params 树 ==========
  ['params']/['modules_actor']/['act']/['bias']    (1,)
  ['params']/['modules_actor']/['act']/['kernel']  (32, 1)
  ['params']/['modules_actor']/['encoder']/['fc1']/['bias'] (64,)
  ['params']/['modules_actor']/['encoder']/['fc1']/['kernel'] (8, 64)
  ['params']/['modules_actor']/['encoder']/['fc2']/['bias'] (32,)
  ['params']/['modules_actor']/['encoder']/['fc2']/['kernel'] (64, 32)
  ['params']/['modules_critic']/['q']/['bias']     (1,)
  ['params']/['modules_critic']/['q']/['kernel']   (34, 1)

========== apply(name='actor') ==========

>>> ModuleDict.__call__  name='actor'  args=((1, 8),)  kwargs_keys=[]
  [Actor.__call__]   收到 obs.shape=(1, 8)
  [Encoder.__call__] 收到 

# 卷积维度变化（对齐 `resnet_v1.py` 的 NHWC 布局）

Flax 特征图 layout：**`(B, H, W, C)`** = batch、高、宽、通道。

`nn.Conv(features, kernel_size, strides)` 会改什么：
- **`features`** → 输出通道 $C_{\text{out}}$（只改最后一维 $C$）
- **`kernel_size`** → 卷积核大小 $(k_H, k_W)$（不直接改输出 shape，但配合 padding/stride 影响 $H,W$）
- **`strides`** → 空间步长 $(s_H, s_W)$（$H,W$ 约变为 $\lceil H/s_H \rceil$，SAME padding 下）

ResNet 残差块里 **shortcut 与主路 shape 不一致** 的两个原因：
1. `strides=(2,2)` → 主路 $H,W$ 减半，shortcut 还是原尺寸
2. `filters` 变大 → 主路 $C$ 变了，shortcut 还是 $C_{\text{in}}$

下面代码逐步打印每一步 shape；跑完对照 `ResNetBlock` 156-169 行。

In [1]:
import functools
import jax
import jax.numpy as jnp
import flax.linen as nn

# Same defaults as resnet_v1.py
conv = functools.partial(
    nn.Conv,
    use_bias=False,
    padding="SAME",
    kernel_init=nn.initializers.kaiming_normal(),
)


def show(tag, x, extra=""):
    b, h, w, c = x.shape
    suffix = f"  |  {extra}" if extra else ""
    print(f"{tag:28s}  shape={str(tuple(x.shape)):20}  (B={b}, H={h}, W={w}, C={c}){suffix}")


def run_conv(x, features, kernel_size, strides=(1, 1), name="conv"):
    layer = conv(features, kernel_size, strides, name=name)
    variables = layer.init(jax.random.key(0), x)
    y = layer.apply(variables, x)
    k = variables["params"]["kernel"]
    show(
        f"Conv {name}",
        y,
        extra=f"kernel={tuple(k.shape)} strides={strides}  # (kH,kW,C_in,C_out)",
    )
    return y


rng = jax.random.key(0)
x0 = jax.random.normal(rng, (2, 8, 8, 16))  # small feature map for easy mental math

print("=" * 72)
print("Part A: one conv — what changes H/W vs C?")
print("=" * 72)
show("input x0", x0)

y_a = run_conv(x0, features=32, kernel_size=(3, 3), strides=(1, 1), name="3x3_s1")
# C: 16 -> 32, H/W unchanged (SAME padding, stride 1)

y_b = run_conv(x0, features=32, kernel_size=(3, 3), strides=(2, 2), name="3x3_s2")
# C: 16 -> 32, H/W: 8 -> 4

y_c = run_conv(x0, features=32, kernel_size=(1, 1), strides=(2, 2), name="1x1_s2")
# projection shortcut style: only mix channels + downsample spatially

print("\n" + "=" * 72)
print("Part B: mini ResNetBlock — when does residual != y ?")
print("=" * 72)

# Case 1: identity shortcut (no conv_proj)
print("\n--- Case 1: strides=(1,1), filters=16 (same as C_in) ---")
x = jax.random.normal(jax.random.key(1), (2, 8, 8, 16))
residual = x
show("residual (= x)", residual)
y = run_conv(x, 16, (3, 3), (1, 1), name="main_conv1")
y = run_conv(y, 16, (3, 3), (1, 1), name="main_conv2")
print(f"residual.shape == y.shape ?  {residual.shape == y.shape}")

# Case 2: stage boundary — channel up + downsample (needs conv_proj)
print("\n--- Case 2: strides=(2,2), filters=32 (C_in=16) ---")
x = jax.random.normal(jax.random.key(2), (2, 8, 8, 16))
residual = x
show("residual (= x)", residual)
y = run_conv(x, 32, (3, 3), (2, 2), name="main_conv1")
y = run_conv(y, 32, (3, 3), (1, 1), name="main_conv2")
show("main path y", y)
print(f"residual.shape == y.shape ?  {residual.shape == y.shape}  -> need 1x1 conv_proj")
residual = run_conv(residual, 32, (1, 1), (2, 2), name="conv_proj")
print(f"after conv_proj, residual.shape == y.shape ?  {residual.shape == y.shape}")
out = residual + y
show("residual + y", out)

print("\n" + "=" * 72)
print("Part C: shape cheat sheet (SAME padding, even H/W)")
print("=" * 72)
for h, s in [(56, 1), (56, 2), (28, 1), (8, 2)]:
    x = jnp.zeros((1, h, h, 64))
    y = run_conv(x, 128, (3, 3), (s, s), name=f"H={h}_s={s}")
    print(f"  spatial: {h} --stride {s}--> {y.shape[1]} , channels: 64 -> {y.shape[-1]}")
    print()

Part A: one conv — what changes H/W vs C?
input x0                      shape=(2, 8, 8, 16)         (B=2, H=8, W=8, C=16)
Conv 3x3_s1                   shape=(2, 8, 8, 32)         (B=2, H=8, W=8, C=32)  |  kernel=(3, 3, 16, 32) strides=(1, 1)  # (kH,kW,C_in,C_out)
Conv 3x3_s2                   shape=(2, 4, 4, 32)         (B=2, H=4, W=4, C=32)  |  kernel=(3, 3, 16, 32) strides=(2, 2)  # (kH,kW,C_in,C_out)
Conv 1x1_s2                   shape=(2, 4, 4, 32)         (B=2, H=4, W=4, C=32)  |  kernel=(1, 1, 16, 32) strides=(2, 2)  # (kH,kW,C_in,C_out)

Part B: mini ResNetBlock — when does residual != y ?

--- Case 1: strides=(1,1), filters=16 (same as C_in) ---
residual (= x)                shape=(2, 8, 8, 16)         (B=2, H=8, W=8, C=16)
Conv main_conv1               shape=(2, 8, 8, 16)         (B=2, H=8, W=8, C=16)  |  kernel=(3, 3, 16, 16) strides=(1, 1)  # (kH,kW,C_in,C_out)
Conv main_conv2               shape=(2, 8, 8, 16)         (B=2, H=8, W=8, C=16)  |  kernel=(3, 3, 16, 16) strides=